In [2]:
import os
import sys

# Memastikan konfigurasi Python worker Spark konsisten
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import count, col

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca data transaksi (CSV) dan data pelanggan (JSON Lines)
df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")

print("Siap untuk latihan.")
print(f"df_transaksi_etl: {df_transaksi_etl.count()} baris")
print(f"df_pelanggan_etl: {df_pelanggan_etl.count()} baris")

Siap untuk latihan.
df_transaksi_etl: 20000 baris
df_pelanggan_etl: 20 baris


---
### Soal 1
> Baca kembali `data_transaksi_besar_parquet` (hasil Sub-bab 6.1.1) menggunakan `spark.read.parquet()`, lalu tampilkan `printSchema()`-nya. Bandingkan dengan skema hasil `inferSchema=True` pada CSV — apakah tipe datanya sama?


In [3]:
# Membaca kembali data Parquet hasil Sub-bab 6.1.1
df_parquet_soal1 = spark.read.parquet("data_transaksi_besar_parquet")

print("=== Skema dari Parquet ===")
df_parquet_soal1.printSchema()

print("\n=== Skema dari CSV (inferSchema=True) ===")
df_transaksi_etl.printSchema()

# Membandingkan tipe data kedua skema
print("\n=== Perbandingan Tipe Data ===")
print("Parquet dtypes:", df_parquet_soal1.dtypes)
print("CSV dtypes    :", df_transaksi_etl.dtypes)
print("Apakah tipe data kedua skema sama persis?", df_parquet_soal1.dtypes == df_transaksi_etl.dtypes)

[Stage 18:>                                                         (0 + 1) / 1]

=== Skema dari Parquet ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)


=== Skema dari CSV (inferSchema=True) ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)


=== Perbandingan Tipe Data ===
Parquet dtypes: [('order_id', 'string'), ('customer_id', 'int'), ('kategori', 'string'), ('kota', 'string'), ('metode_pembayaran', 'string'), ('pendapatan', 'int')]
CSV dtypes    : [('order_id', 'string'), ('customer_id', 'int'), ('kategori', 'string'), ('kota', 'string'), ('metode_pembayaran', 'string'), ('pendapatan', 'int')]
Apakah tipe data kedua skema sama persis? True


---
### Soal 2
> Gabungkan `df_transaksi_etl` dengan `df_pelanggan_etl`, lalu tampilkan jumlah transaksi untuk masing-masing nilai `membership` (`Silver`/`Gold`/`Platinum`).


In [4]:
# Menggabungkan data transaksi dengan data pelanggan berdasarkan customer_id
df_gabung_soal2 = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="inner")

# Menghitung jumlah transaksi untuk masing-masing membership
df_jumlah_per_membership = df_gabung_soal2.groupBy("membership").agg(
    count("order_id").alias("jumlah_transaksi")
).orderBy(col("jumlah_transaksi").desc())

print("=== Jumlah Transaksi per Nilai Membership ===")
df_jumlah_per_membership.show()

=== Jumlah Transaksi per Nilai Membership ===


[Stage 20:>                                                         (0 + 1) / 1]

+----------+----------------+
|membership|jumlah_transaksi|
+----------+----------------+
|    Silver|            8994|
|      Gold|            6990|
|  Platinum|            4016|
+----------+----------------+



---
### Soal 3
> Tulis ulang hasil gabungan Soal 2 (sebelum di-`groupBy`) ke disk lokal dalam format Parquet, dipartisi berdasarkan `membership`. Tampilkan struktur foldernya dengan `!ls`.


In [5]:
# Menulis ulang hasil gabungan (df_gabung_soal2) ke disk lokal dalam format Parquet dipartisi berdasarkan membership
output_path_soal3 = "data_transaksi_partisi_membership"
df_gabung_soal2.write.mode("overwrite").partitionBy("membership").parquet(output_path_soal3)

print(f"Data berhasil ditulis ke '{output_path_soal3}' dengan partisi membership.\n")
print("Struktur folder yang dihasilkan:")
!ls -la {output_path_soal3}

Data berhasil ditulis ke 'data_transaksi_partisi_membership' dengan partisi membership.

Struktur folder yang dihasilkan:
total 24
drwxr-xr-x 5 raka raka 4096 Sep 27 15:34  .
drwxrwxr-x 6 raka raka 4096 Sep 27 15:34  ..
drwxr-xr-x 2 raka raka 4096 Sep 27 15:34 'membership=Gold'
drwxr-xr-x 2 raka raka 4096 Sep 27 15:34 'membership=Platinum'
drwxr-xr-x 2 raka raka 4096 Sep 27 15:34 'membership=Silver'
-rw-r--r-- 1 raka raka    0 Sep 27 15:34  _SUCCESS
-rw-r--r-- 1 raka raka    8 Sep 27 15:34  ._SUCCESS.crc


---
### Soal 4 (Refleksi Singkat)
> Dalam 2-3 kalimat: mengapa partitioning berdasarkan kolom `kota` cocok untuk skenario kita, namun akan menjadi **pilihan buruk** jika diterapkan pada kolom seperti `order_id` yang nilainya selalu unik di setiap baris? Tulis jawaban pada markdown cell di bawah ini.

**Jawaban:**
Partitioning berdasarkan kolom `kota` sangat cocok untuk skenario kita karena memiliki **kardinalitas rendah** (hanya 5 nilai unik), sehingga menghasilkan sedikit sub-folder dengan distribusi ukuran data yang seimbang serta mengaktifkan optimasi ***partition pruning*** yang mempercepat pemfilteran query tanpa membaca seluruh data. Sebaliknya, partitioning pada kolom ber-kardinalitas unik seperti `order_id` (20.000 nilai berbeda di setiap baris) akan menjadi **pilihan yang sangat buruk** karena memicu fenomena ***small files problem***, di mana Spark akan membuat 20.000 direktori partisi kecil yang masing-masing hanya memuat satu file mini. Hal ini membebani penyimpanan metadata NameNode/sistem berkas secara masif, merusak efisiensi kompresi kolumnar Parquet, dan menyebabkan degradasi performa I/O yang sangat parah saat dibaca kembali.


---
## Menutup SparkSession


In [6]:
spark.stop()
print("SparkSession berhasil ditutup secara aman.")

SparkSession berhasil ditutup secara aman.
